
# Consumption–Saving RL Project — Risk-Aversion Robustness

## Research question
위험회피도 \(\sigma\)가 커질수록 최적 소비·저축 정책은 어떻게 바뀌는가?

\[
\sigma \in \{1.5,\;2.0,\;3.0\}
\]

다른 calibration은 모두 고정합니다.

- \(R=1.00985\)
- \(\beta=0.9703\)
- \(y_H=1.0,\;y_L=0.472\)
- 동일한 income Markov transition
- 동일한 20개 saving-share action
  \[
  z\in\{0,0.05,\ldots,0.95\}
  \]

### Important
서로 다른 \(\sigma\)의 CRRA utility 값 자체는 척도가 다르므로 직접 비교하지 않습니다.

비교 대상:
1. optimal saving-share policy
2. \(a=1\)에서 High/Low income별 saving share
3. long-run mean asset
4. long-run mean saving share
5. long-run mean consumption


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

R = 1.00985
BETA = 0.9703

Y_HIGH = 1.0
Y_LOW = 0.472

P_INCOME = np.array([
    [0.9393, 0.0607],
    [0.3920, 0.6080]
], dtype=np.float64)

SAVING_SHARES = np.arange(0.0, 1.0, 0.05, dtype=np.float64)

A_MAX = 4.5
N_ASSETS = 500
SIGMAS = [1.5, 2.0, 3.0]

print("Setup loaded.")


In [ ]:

def solve_dp_matched_sigma(sigma, tolerance=1e-10, max_iterations=5000):

    asset_grid = np.linspace(0.0, A_MAX, N_ASSETS)
    V = np.zeros((N_ASSETS, 2), dtype=np.float64)
    policy_share = np.zeros_like(V)

    for iteration in range(max_iterations):

        V_new = np.empty_like(V)

        for income_state, income in [(0, Y_HIGH), (1, Y_LOW)]:

            cash = R * asset_grid + income

            Q = np.full(
                (N_ASSETS, len(SAVING_SHARES)),
                -np.inf,
                dtype=np.float64
            )

            for j, z in enumerate(SAVING_SHARES):

                next_asset = z * cash
                consumption = (1.0 - z) * cash

                feasible = (
                    (consumption > 1e-12)
                    &
                    (next_asset <= A_MAX)
                )

                V_H = np.interp(next_asset, asset_grid, V[:, 0])
                V_L = np.interp(next_asset, asset_grid, V[:, 1])

                expected_future = (
                    P_INCOME[income_state, 0] * V_H
                    +
                    P_INCOME[income_state, 1] * V_L
                )

                if np.isclose(sigma, 1.0):
                    utility = np.log(consumption)
                else:
                    utility = (
                        consumption ** (1.0 - sigma)
                        /
                        (1.0 - sigma)
                    )

                q = utility + BETA * expected_future
                q[~feasible] = -np.inf
                Q[:, j] = q

            best_idx = np.argmax(Q, axis=1)

            V_new[:, income_state] = (
                Q[np.arange(N_ASSETS), best_idx]
            )

            policy_share[:, income_state] = (
                SAVING_SHARES[best_idx]
            )

        diff = np.max(np.abs(V_new - V))
        V = V_new

        if diff < tolerance:
            break

    print(
        f"sigma={sigma}: converged after {iteration + 1} iterations"
    )

    return asset_grid, V, policy_share


solutions = {}

for sigma in SIGMAS:
    grid, V, policy = solve_dp_matched_sigma(sigma)

    solutions[sigma] = {
        "asset_grid": grid,
        "V": V,
        "policy_share": policy
    }

print("All sigma cases solved.")


In [ ]:

plt.figure(figsize=(10, 5))

for sigma in SIGMAS:
    plt.plot(
        solutions[sigma]["asset_grid"],
        solutions[sigma]["policy_share"][:, 0],
        label=f"sigma = {sigma}"
    )

plt.xlabel("Current Asset")
plt.ylabel("Optimal Saving Share")
plt.title("Risk Aversion and Optimal Saving Policy: High Income")
plt.legend()
plt.show()


In [ ]:

plt.figure(figsize=(10, 5))

for sigma in SIGMAS:
    plt.plot(
        solutions[sigma]["asset_grid"],
        solutions[sigma]["policy_share"][:, 1],
        label=f"sigma = {sigma}"
    )

plt.xlabel("Current Asset")
plt.ylabel("Optimal Saving Share")
plt.title("Risk Aversion and Optimal Saving Policy: Low Income")
plt.legend()
plt.show()


In [ ]:

representative_assets = [0.25, 0.50, 1.00, 2.00, 3.00, 4.00]

rows = []

for sigma in SIGMAS:

    grid = solutions[sigma]["asset_grid"]
    policy = solutions[sigma]["policy_share"]

    for asset in representative_assets:

        idx = np.argmin(
            np.abs(grid - asset)
        )

        rows.append({
            "sigma": sigma,
            "asset": grid[idx],
            "High-income saving share": policy[idx, 0],
            "Low-income saving share": policy[idx, 1]
        })

policy_table = pd.DataFrame(rows)
display(policy_table)


In [ ]:

initial_rows = []

for sigma in SIGMAS:

    grid = solutions[sigma]["asset_grid"]
    policy = solutions[sigma]["policy_share"]

    idx = np.argmin(
        np.abs(grid - 1.0)
    )

    initial_rows.append({
        "sigma": sigma,
        "asset_grid_value": grid[idx],
        "Saving share | High income": policy[idx, 0],
        "Saving share | Low income": policy[idx, 1]
    })

initial_policy_table = pd.DataFrame(initial_rows)
display(initial_policy_table)



## Long-run simulation

각 \(\sigma\)의 optimal DP policy를 동일한 income-risk 환경에서 시뮬레이션합니다.

- 5,000 agents
- 1,000 quarters
- first 300 quarters = burn-in
- initial asset = 1
- initial income = High
- same random seed for every \(\sigma\)

Utility가 아니라 경제적 행동을 비교합니다.


In [ ]:

def nearest_policy_action(
    assets,
    income_states,
    asset_grid,
    policy_share
):

    idx = np.clip(
        np.searchsorted(asset_grid, assets),
        1,
        len(asset_grid) - 1
    )

    left = asset_grid[idx - 1]
    right = asset_grid[idx]

    use_right = (
        np.abs(assets - right)
        <
        np.abs(assets - left)
    )

    nearest_idx = np.where(
        use_right,
        idx,
        idx - 1
    )

    return policy_share[
        nearest_idx,
        income_states
    ]


def simulate_long_run(
    asset_grid,
    policy_share,
    n_agents=5000,
    horizon=1000,
    burn_in=300,
    seed=20261102
):

    rng = np.random.default_rng(seed)

    assets = np.ones(n_agents)
    income_states = np.zeros(n_agents, dtype=int)

    asset_sum = 0.0
    saving_sum = 0.0
    consumption_sum = 0.0
    observations = 0

    for t in range(horizon):

        incomes = np.where(
            income_states == 0,
            Y_HIGH,
            Y_LOW
        )

        z = nearest_policy_action(
            assets,
            income_states,
            asset_grid,
            policy_share
        )

        cash = R * assets + incomes
        consumption = (1.0 - z) * cash
        next_assets = z * cash

        if t >= burn_in:

            asset_sum += assets.sum()
            saving_sum += z.sum()
            consumption_sum += consumption.sum()
            observations += n_agents

        p_low = np.where(
            income_states == 0,
            P_INCOME[0, 1],
            P_INCOME[1, 1]
        )

        income_states = (
            rng.random(n_agents) < p_low
        ).astype(int)

        assets = next_assets

    return {
        "Mean asset": asset_sum / observations,
        "Mean saving share": saving_sum / observations,
        "Mean consumption": consumption_sum / observations
    }


simulation_rows = []

for sigma in SIGMAS:

    result = simulate_long_run(
        solutions[sigma]["asset_grid"],
        solutions[sigma]["policy_share"]
    )

    simulation_rows.append({
        "sigma": sigma,
        **result
    })

long_run_table = pd.DataFrame(simulation_rows)
display(long_run_table)


In [ ]:

plt.figure(figsize=(8, 5))
plt.plot(
    long_run_table["sigma"],
    long_run_table["Mean asset"],
    marker="o"
)
plt.xlabel("Risk Aversion (sigma)")
plt.ylabel("Long-Run Mean Asset")
plt.title("Risk Aversion and Long-Run Asset Holdings")
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(
    long_run_table["sigma"],
    long_run_table["Mean saving share"],
    marker="o"
)
plt.xlabel("Risk Aversion (sigma)")
plt.ylabel("Long-Run Mean Saving Share")
plt.title("Risk Aversion and Long-Run Saving")
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(
    long_run_table["sigma"],
    long_run_table["Mean consumption"],
    marker="o"
)
plt.xlabel("Risk Aversion (sigma)")
plt.ylabel("Long-Run Mean Consumption")
plt.title("Risk Aversion and Long-Run Consumption")
plt.show()


In [ ]:

summary = initial_policy_table.merge(
    long_run_table,
    on="sigma",
    how="left"
)

display(summary)

print()
print("Interpretation guide")
print("--------------------")
print("1) Do NOT compare value-function levels across sigma.")
print("2) Focus on saving policy, long-run assets, saving, and consumption.")
print("3) Higher sigma need not raise saving at every single state.")
print("4) CRRA changes both risk aversion and intertemporal substitution.")
print("5) Use the overall policy shift and long-run behavior for the final conclusion.")
